In [ ]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score


DATA_PATH = "/Users/hd/Desktop/Machine Learning/ML-Excercises/train.csv"

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u200b", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

clf = LinearSVC(C=1.0)

runs = 0
f1_total = 0.0
scores = []

for seed in range(100):

    train = pd.read_csv(DATA_PATH, sep=",")

    if "targets" in train.columns and "label" not in train.columns:
        train = train.rename(columns={"targets": "label"})

    test = train.sample(frac=0.05, random_state=seed)
    train = train.drop(test.index)

    y_train = np.array(train["label"])
    y_test  = np.array(test["label"])

    X_train_raw = train["samples"].map(clean_text)
    X_test_raw  = test["samples"].map(clean_text)

    X_train = tfidf.fit_transform(X_train_raw)
    X_test  = tfidf.transform(X_test_raw)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    f1 = f1_score(y_test, y_pred, average="macro")
    scores.append(f1)
    f1_total += f1
    runs += 1

scores = np.array(scores)
print(f"Average F1 after {runs} runs: {f1_total/runs:.6f}")
print(f"Std dev: {scores.std():.6f}")
print(f"Min / Max: {scores.min():.6f} / {scores.max():.6f}")

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# =========================
# EDIT THESE TWO
# =========================
NAME = "Daniel_Tesfai_Kebede"
MATRIKELNUMMER = "1716694"

# =========================
# PATHS (edit if needed)
# =========================
TRAIN_PATH = "/Users/hd/Desktop/Machine Learning/ML-Excercises/train.csv"
HOLDBACK_PATH = "/Users/hd/Desktop/Machine Learning/ML-Excercises/holdback_no_targets.csv"
OUT_PATH = f"/Users/hd/Desktop/Machine Learning/ML-Excercises/predictions_{NAME}_{MATRIKELNUMMER}.csv"

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u200b", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -------------------------
# 1) Load train
# -------------------------
train = pd.read_csv(TRAIN_PATH, sep=",")

# Your baseline uses "targets" in train.csv; keep it for training too.
if "targets" not in train.columns or "samples" not in train.columns:
    raise ValueError(f"train.csv must contain columns ['id','samples','targets'] (id optional). Found: {list(train.columns)}")

train = train.dropna(subset=["samples", "targets"]).reset_index(drop=True)
train["samples"] = train["samples"].map(clean_text)
y = train["targets"].astype(int).values
X = train["samples"].values

bad = sorted(set(np.unique(y)) - {0, 1, 2})
if bad:
    raise ValueError(f"Unexpected labels in train.csv targets: {bad}. Allowed: [0,1,2]")

# -------------------------
# Train model
# -------------------------
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

clf = LinearSVC(C=1.0)

X_vec = tfidf.fit_transform(X)
clf.fit(X_vec, y)

# -------------------------
# Load holdback + predict
# -------------------------
hold = pd.read_csv(HOLDBACK_PATH, sep=",")

if "id" not in hold.columns or "samples" not in hold.columns:
    raise ValueError(f"holdback_no_targets.csv must contain columns ['id','samples']. Found: {list(hold.columns)}")

hold = hold.dropna(subset=["id", "samples"]).reset_index(drop=True)
hold["samples"] = hold["samples"].map(clean_text)

X_hold = tfidf.transform(hold["samples"].values)
pred = clf.predict(X_hold).astype(int)

# -------------------------
# Build submission (MUST be id,targets)
# -------------------------
submission = pd.DataFrame({
    "id": hold["id"].astype(int),
    "targets": pred
})


# Save
submission.to_csv(OUT_PATH, sep=",", index=False)

print("Wrote submission:", OUT_PATH)
print("First 5 rows:\n", submission.head())


In [17]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# =========================
# EDIT THESE
# =========================
NAME = "Daniel_Tesfai_Kebede"
MATRIKELNUMMER = "1716694"
EVALUATE = True      # ← set False right before final submission if you want silence

# =========================
# PATHS
# =========================
TRAIN_PATH = "/Users/hd/Desktop/Machine Learning/ML-Excercises/train.csv"
HOLDBACK_PATH = "/Users/hd/Desktop/Machine Learning/ML-Excercises/holdback_no_targets.csv"
OUT_PATH = f"/Users/hd/Desktop/Machine Learning/ML-Excercises/predictions_{NAME}_{MATRIKELNUMMER}.csv"

# -------------------------
# Cleaning
# -------------------------
def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u200b", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -------------------------
# Load train data
# -------------------------
train = pd.read_csv(TRAIN_PATH, sep=",")
train = train.dropna(subset=["samples", "targets"]).reset_index(drop=True)
train["samples"] = train["samples"].map(clean_text)

X_all = train["samples"].values
y_all = train["targets"].astype(int).values

# -------------------------
# Model
# -------------------------
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
clf = LinearSVC(C=1.0)

# ============================================================
# 1) EVALUATION MODE (prints F1 – FOR YOU ONLY)
# ============================================================
if EVALUATE:
    scores = []

    for seed in range(100):
        # stratified 5% split
        test_parts = []
        for label, grp in train.groupby("targets"):
            n = max(1, int(round(len(grp) * 0.05)))
            test_parts.append(grp.sample(n=n, random_state=seed))

        test = pd.concat(test_parts)
        tr = train.drop(test.index)

        X_tr = tfidf.fit_transform(tr["samples"])
        y_tr = tr["targets"].values
        X_te = tfidf.transform(test["samples"])
        y_te = test["targets"].values

        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)

        scores.append(f1_score(y_te, pred, average="macro"))

    scores = np.array(scores)
    print("=== LOCAL EVALUATION ===")
    print(f"Average F1 after 100 runs: {scores.mean():.6f}")
    print(f"Std dev: {scores.std():.6f}")
    print(f"Min / Max: {scores.min():.6f} / {scores.max():.6f}")
    print("========================\n")

# ============================================================
# 2) SUBMISSION MODE (always runs)
# ============================================================

# Train on FULL train.csv
X_vec = tfidf.fit_transform(X_all)
clf.fit(X_vec, y_all)

# Load holdback
hold = pd.read_csv(HOLDBACK_PATH, sep=",")
hold = hold.dropna(subset=["id", "samples"]).reset_index(drop=True)
hold["samples"] = hold["samples"].map(clean_text)

# Predict
pred = clf.predict(tfidf.transform(hold["samples"].values))

# Export EXACT checker format
submission = pd.DataFrame({
    "id": hold["id"].astype(int),
    "targets": pred.astype(int)
})

submission.to_csv(OUT_PATH, sep=",", index=False)

print("Submission written:", OUT_PATH)
print(submission.head())

=== LOCAL EVALUATION ===
Average F1 after 100 runs: 0.611702
Std dev: 0.063742
Min / Max: 0.425456 / 0.796771

Submission written: /Users/hd/Desktop/Machine Learning/ML-Excercises/predictions_Daniel_Tesfai_Kebede_1716694.csv
    id  targets
0  433        2
1  453        2
2  535        1
3  679        0
4  199        2
